In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle
import os

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('/home/dabiyyu/projects/data-scientist/churn-prediction/data/Telco_customer_churn.csv', delimiter=";")

# Fix comma decimal separators
for col in df.columns:
    converted = (df[col].astype(str)
                 .str.replace(',', '.')
                 .str.strip()
                 .pipe(pd.to_numeric, errors='coerce'))
    success_rate = converted.notna().sum() / len(df)
    if success_rate > 0.8:
        df[col] = converted

# Fix Total Charges remaining nulls
df['Total Charges'] = df['Total Charges'].fillna(0)

print("Shape:", df.shape)
print("Nulls:\n", df.isnull().sum()[df.isnull().sum() > 0])

Shape: (7043, 33)
Nulls:
 Churn Reason    5174
dtype: int64


In [13]:
cols_to_drop = [
    'CustomerID',     # identifier, not predictive
    'Count',          # constant
    'Country',        # constant (all US)
    'State',          # low variance
    'Lat Long',       # redundant with Latitude/Longitude
    'City',           # too many unique values → explodes encoding
    'Zip Code',       # same issue
    'Latitude',       # raw coordinates not useful for this model
    'Longitude',      # same
    'Churn Reason',   # only exists for churned customers, causes leakage
    'Churn Score',    # pre-calculated score, causes leakage
    'Churn Label',    # text version of target, keep Churn Value instead
]

# Only drop columns that exist in the dataframe
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=cols_to_drop)

print("Shape after dropping:", df.shape)
print("Remaining columns:")
print(df.columns.tolist())

Shape after dropping: (7043, 21)
Remaining columns:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value', 'CLTV']


In [14]:
target = 'Churn Value'

X = df.drop(columns=[target])
y = df[target]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Target distribution:\n", y.value_counts())

Features shape: (7043, 20)
Target shape: (7043,)
Target distribution:
 Churn Value
0    5174
1    1869
Name: count, dtype: int64


In [15]:
# Separate columns by type for different encoding strategies
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical columns:", len(categorical_cols))
print(categorical_cols)

print("\nNumerical columns:", len(numerical_cols))
print(numerical_cols)

Categorical columns: 16
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']

Numerical columns: 4
['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']


In [16]:
# Columns with only 2 unique values → LabelEncoder
binary_cols = [col for col in categorical_cols 
               if X[col].nunique() == 2]

print("Binary columns to encode:")
print(binary_cols)

le = LabelEncoder()
for col in binary_cols:
    X[col] = le.fit_transform(X[col])
    print(f"  {col}: {X[col].unique()}")

Binary columns to encode:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Paperless Billing']
  Gender: [1 0]
  Senior Citizen: [0 1]
  Partner: [0 1]
  Dependents: [0 1]
  Phone Service: [1 0]
  Paperless Billing: [1 0]


In [17]:
# Columns with more than 2 unique values → One-Hot Encoding
multi_cols = [col for col in categorical_cols 
              if X[col].nunique() > 2]

print("Multi-category columns to one-hot encode:")
for col in multi_cols:
    print(f"  {col}: {X[col].unique()}")

X = pd.get_dummies(X, columns=multi_cols, drop_first=True)

print("\nShape after encoding:", X.shape)
print("New columns added:")
print(X.columns.tolist())

Multi-category columns to one-hot encode:
  Multiple Lines: ['No' 'Yes' 'No phone service']
  Internet Service: ['DSL' 'Fiber optic' 'No']
  Online Security: ['Yes' 'No' 'No internet service']
  Online Backup: ['Yes' 'No' 'No internet service']
  Device Protection: ['No' 'Yes' 'No internet service']
  Tech Support: ['No' 'Yes' 'No internet service']
  Streaming TV: ['No' 'Yes' 'No internet service']
  Streaming Movies: ['No' 'Yes' 'No internet service']
  Contract: ['Month-to-month' 'Two year' 'One year']
  Payment Method: ['Mailed check' 'Electronic check' 'Bank transfer (automatic)'
 'Credit card (automatic)']

Shape after encoding: (7043, 31)
New columns added:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Paperless Billing', 'Monthly Charges', 'Total Charges', 'CLTV', 'Multiple Lines_No phone service', 'Multiple Lines_Yes', 'Internet Service_Fiber optic', 'Internet Service_No', 'Online Security_No internet service', 'Online Security_Yes', 

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y      # preserve churn ratio in both splits
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nChurn rate in train:", y_train.mean().round(3))
print("Churn rate in test:", y_test.mean().round(3))

Train size: (5634, 31)
Test size: (1409, 31)

Churn rate in train: 0.265
Churn rate in test: 0.265


In [19]:
# Only scale numerical columns — not the encoded binary columns
scaler = StandardScaler()

# Get numerical col names that still exist after encoding
num_cols_to_scale = [col for col in numerical_cols if col in X_train.columns]
print("Columns to scale:", num_cols_to_scale)

X_train[num_cols_to_scale] = scaler.fit_transform(X_train[num_cols_to_scale])
X_test[num_cols_to_scale] = scaler.transform(X_test[num_cols_to_scale])

print("\nScaling complete")
print("Sample scaled values:")
print(X_train[num_cols_to_scale].describe().round(2))

Columns to scale: ['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']

Scaling complete
Sample scaled values:
       Tenure Months  Monthly Charges  Total Charges     CLTV
count        5634.00          5634.00        5634.00  5634.00
mean           -0.00            -0.00           0.00     0.00
std             1.00             1.00           1.00     1.00
min            -1.32            -1.54          -1.01    -2.03
25%            -0.96            -0.97          -0.83    -0.79
50%            -0.14             0.18          -0.40     0.10
75%             0.92             0.83           0.67     0.83
max             1.61             1.79           2.80     1.77


In [20]:
# Save train/test sets
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

# Save scaler and feature names for use in Streamlit app later
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../models/feature_names.pkl', 'wb') as f:
    pickle.dump(X_train.columns.tolist(), f)

print("Saved:")
print("  ../data/X_train.csv")
print("  ../data/X_test.csv")
print("  ../data/y_train.csv")
print("  ../data/y_test.csv")
print("  ../models/scaler.pkl")
print("  ../models/feature_names.pkl")

Saved:
  ../data/X_train.csv
  ../data/X_test.csv
  ../data/y_train.csv
  ../data/y_test.csv
  ../models/scaler.pkl
  ../models/feature_names.pkl


In [21]:
# Check which columns created so many dummies
print("Columns with most unique values:")
original_X = df.drop(columns=[target])
for col in original_X.select_dtypes(include='object').columns:
    print(f"  {col}: {original_X[col].nunique()} unique values")

Columns with most unique values:
  Gender: 2 unique values
  Senior Citizen: 2 unique values
  Partner: 2 unique values
  Dependents: 2 unique values
  Phone Service: 2 unique values
  Multiple Lines: 3 unique values
  Internet Service: 3 unique values
  Online Security: 3 unique values
  Online Backup: 3 unique values
  Device Protection: 3 unique values
  Tech Support: 3 unique values
  Streaming TV: 3 unique values
  Streaming Movies: 3 unique values
  Contract: 3 unique values
  Paperless Billing: 2 unique values
  Payment Method: 4 unique values
